In [0]:
class Silver_race_results():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze"
    silver_path = "formula1_race_project/silver"

    def __init__(self,table,drivers_df,constructors_df,circuits_df): #
         self.table=table
         self.drivers_df=drivers_df
         self.constructors_df=constructors_df
         self.circuits_df=circuits_df

    def max_watermark_value(self):
        from pyspark.sql.functions import max,col
        #fetching max_ingestion_date from gold layer table race_results
        if spark.catalog.tableExists("formula1_race.silver.race_results"):
            results_max_ingestion_date =spark.read.table('formula1_race.silver.race_results').agg(max(col('results_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
            if results_max_ingestion_date is None:
                results_max_ingestion_date='1900-01-01 00:00:00'
        else:
            results_max_ingestion_date='1900-01-01 00:00:00'
        print(f"results_max_ingestion_date:{results_max_ingestion_date}") #printing last water mark ingetsion timestamp
        print(f"results_max_ingestion_date:",type(results_max_ingestion_date))
        return results_max_ingestion_date
            
    def read_input(self,list_max_ingest):
        from pyspark.sql.functions import max,col,expr,count
        results_max_ingestion_date=list_max_ingest
    
        #fetching incremental result data 
        incr_results_df= (spark.read.table('formula1_race.silver.results')
                        .filter(col('results_ingestion_date')>results_max_ingestion_date)
                            )
        
        #Printing incremental result data records count
        print("incr_results_df")
        display(incr_results_df.select(count(col('race_id'))))
       
        #fetching distinct race_id from incremental result data and printing count of records
        incr_results_df_list= incr_results_df.select(col("race_id").alias("result_race_id")).distinct()
        print("incr_results_df_list")
        display(incr_results_df_list.select(count(col('result_race_id'))))
        
        #fetching only requeried data from race table which is required for joining incremental result data and printing the count of records
        races_df=spark.read.table('formula1_race.silver.races')
        incr_race_df=races_df.join(incr_results_df_list,races_df["race_id"]==incr_results_df_list['result_race_id'],'inner').drop(col('result_race_id'))
        print("incr_race_df")
        display(incr_race_df.select(count(col('race_id'))))

        # drivers_df=spark.read.table('formula1_race.silver.drivers')
        # constructors_df=spark.read.table('formula1_race.silver.constructors')
        # circuits_df=spark.read.table('formula1_race.silver.circuits')
        #passing all requeried tables for join as list
        read_df_list=[incr_race_df,incr_results_df,self.drivers_df,self.constructors_df,self.circuits_df]
        return read_df_list
    
    def apply_transformations(self,read_df_list):
        from pyspark.sql.functions import round,col,broadcast,expr,count
        races_df = read_df_list[0]
        result_df = read_df_list[1]
        drivers_df=read_df_list[2]
        constructors_df=read_df_list[3]
        circuits_df=read_df_list[4]

        #performing join operation only on  incremental result data with all other tables
        race_results_df= (result_df.join(races_df,
           [result_df["race_id"] == races_df["race_id"]],'inner')
                .join(drivers_df,["driver_id" ],'inner')
                .join(constructors_df,["constructor_id"],'inner')
                .join(circuits_df,["circuit_id"],'inner')
                .selectExpr("race_year","race_name","circuit_name" ,"circuit_country","circuit_location","latitude","longitude","driver_name","driver_nationality", "constructor_team","constructor_nationality","grid","result_position","result_position_order","result_position_text","result_points","laps as result_laps","time as  result_time","fastest_lap as result_fastest_lap","fastest_lap_rank  as result_fastest_lap_rank","fastest_lap_time as result_fastest_lap_time","fastest_lap_speed as result_fastest_lap_speed","results_ingestion_date"
                                   )
                        )
        print("detailed execution plain")
        race_results_df.explain(True)
        #displaying resulted data records count
        display(race_results_df.select(count("*")))
        return race_results_df

        
    def write_output(self,apply_tran_df):
        # writing those data into gold layer table by partitioning according to filter approach using append mode
        (apply_tran_df.write.partitionBy("race_year","race_name")
         .mode("append")
         .saveAsTable(f"formula1_race.silver.{self.table}"))
        display(spark.sql(f'select count(*) from formula1_race.silver.{self.table}'))
        print("Data write into silver race_results table is Done")
    
    def process(self):
        print("Started Sliver-ingestion-race_results  is running....")
        results_watermark=self.max_watermark_value() # return latest water mark value
        read_df_list=self.read_input(results_watermark) #return list of all required tables
        apply_tran_df=self.apply_transformations(read_df_list) #returns joined table
        self.write_output(apply_tran_df) #write data into gold table
        return results_watermark

In [0]:
# Silver_race_results_instance = Silver_race_results("race_results")
# race_results_max_ingestion=Silver_race_results_instance .process()
# print("Successfully Silver_race_results is ran")

In [0]:
# dbutils.jobs.taskValues.set(key='rr_water_mark', value=str(race_results_max_ingestion))